# Empirical-prior and support sensitivity

The single-round NPE training prior is the simulation’s empirical mass distribution. This makes the posterior and its biases depend on that population.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'pyproject.toml').is_file())
def table(relative):
    return pd.read_csv(ROOT / 'results' / relative)
def saved_json(relative):
    return json.loads((ROOT / 'results' / relative).read_text(encoding='utf-8'))
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_columns', 16)

### Empirical training-prior swaps

With a fixed likelihood mapping, a diagnostic prior swap reweights saved samples by
$w(\theta) \propto p_{\rm destination}(\theta)/p_{\rm source}(\theta)$.
This is an assumption-dependent sensitivity, not a new trained posterior or proof that
the likelihood is shared. Training-only KDEs use bandwidth factors 0.5, 1 and 2.
Low effective sample size or a dominant normalized weight makes reweighting unreliable.
Combined training uses an equal-domain empirical prior mixture.

For the Local Group observational-input mixture, each conditional component must be
reweighted and normalized separately before averaging. Globally reweighting flattened
draws would change the weights on observational realizations.


In [2]:
display(table('prior/training_prior_summary.csv'))
display(table('prior/density_estimation_details.csv'))
prior=table('prior/prior_swap_main_table.csv')
display(prior[['model','evaluation','treatment','rmse','bias','coverage68','ess_median','ess_min','fraction_ess_below50']])
sensitivity=table('prior/prior_swap_test_metrics.csv')
display(sensitivity.query("subset == 'pooled'")[['model','bandwidth_factor','treatment','bias','rmse','ess_min','max_normalized_weight']])
display(table('prior/local_group_prior_swap.csv').head(16))

,domain,n,mean,median,std,minimum,maximum
0,flamingo,109150,12.446285,12.400646,0.314523,11.544068,13.864985
1,tng,1169,12.704523,12.638891,0.354643,11.959418,13.865908


,domain,bandwidth_factor,kernel_std,normalizer,max_log_interpolation_error
0,flamingo,0.5,0.015453,1.000000,1.037283e-04
1,tng,0.5,0.043172,1.000000,1.257517e-05
2,flamingo,1.0,0.030906,0.999998,2.517041e-05
3,tng,1.0,0.086343,1.000000,3.110472e-06
4,flamingo,2.0,0.061813,0.999991,6.134170e-06
5,tng,2.0,0.172687,0.999986,7.221152e-07


,model,evaluation,treatment,rmse,bias,coverage68,ess_median,ess_min,fraction_ess_below50
0,flamingo_dynamics,tng,original,0.365471,-0.155426,0.649351,500.000000,500.000000,0.000000
1,flamingo_dynamics,tng,prior_swap,0.356371,0.007598,0.662338,365.028767,1.000000,0.151515
2,flamingo_stellar_sum,tng,original,0.360156,-0.316399,0.199134,500.000000,500.000000,0.000000
3,flamingo_stellar_sum,tng,prior_swap,0.307388,-0.248986,0.316017,396.041576,189.241130,0.000000
4,tng_dynamics,flamingo,original,0.334127,0.194850,0.604559,500.000000,500.000000,0.000000
5,tng_dynamics,flamingo,prior_swap,0.495619,-0.292394,0.545658,113.903450,1.034975,0.434350
6,tng_stellar_sum,flamingo,original,0.357790,0.321360,0.100892,500.000000,500.000000,0.000000
7,tng_stellar_sum,flamingo,prior_swap,0.315789,0.272424,0.181793,457.916860,1.724434,0.003176
8,combined_dynamics,flamingo,original,0.262693,0.087176,0.747256,500.000000,500.000000,0.000000
9,combined_dynamics,flamingo,prior_swap,0.236681,0.001034,0.789574,462.058894,298.647774,0.000000


,model,bandwidth_factor,treatment,bias,rmse,ess_min,max_normalized_weight
0,flamingo_dynamics,0.0,original,-0.155426,0.365471,500.000000,0.002000
3,flamingo_dynamics,0.5,prior_swap,0.017723,0.366154,1.000000,1.000000
6,flamingo_dynamics,1.0,prior_swap,0.007598,0.356371,1.000000,1.000000
9,flamingo_dynamics,2.0,prior_swap,-0.003712,0.354360,1.000000,1.000000
12,flamingo_stellar_sum,0.0,original,-0.316399,0.360156,500.000000,0.002000
15,flamingo_stellar_sum,0.5,prior_swap,-0.242551,0.303935,151.963130,0.057843
18,flamingo_stellar_sum,1.0,prior_swap,-0.248986,0.307388,189.241130,0.053099
21,flamingo_stellar_sum,2.0,prior_swap,-0.266935,0.319118,228.548574,0.046176
24,tng_dynamics,0.0,original,0.194850,0.334127,500.000000,0.002000
27,tng_dynamics,0.5,prior_swap,-0.437116,0.608240,1.000000,1.000000


,model,source_prior,target_prior,scenario,bandwidth_factor,original_log_median,swapped_log_median,median_shift_dex,...,swapped_width68,swapped_mass_q16,swapped_mass_median,swapped_mass_q84,proposals_per_component,ess_component_median,ess_component_p05,ess_component_min
0,flamingo_dynamics,flamingo,tng,nominal,0.5,12.471439,12.536019,0.064580,...,0.312640,2.415343,3.435729,4.961572,100000,86894.933396,86894.933396,86894.933396
1,flamingo_dynamics,flamingo,tng,nominal,1.0,12.471439,12.537284,0.065844,...,0.318838,2.402448,3.445749,5.006013,100000,87312.486676,87312.486676,87312.486676
2,flamingo_dynamics,flamingo,tng,nominal,2.0,12.471439,12.539561,0.068122,...,0.338915,2.345333,3.463869,5.118225,100000,88201.666333,88201.666333,88201.666333
3,flamingo_dynamics,flamingo,tng,uncertainty,0.5,12.478419,12.540059,0.061640,...,0.322826,2.420301,3.467843,5.089743,20,17.584406,15.812124,8.630926
4,flamingo_dynamics,flamingo,tng,uncertainty,1.0,12.478419,12.541225,0.062806,...,0.328845,2.407942,3.477163,5.134412,20,17.656249,16.055416,8.731483
5,flamingo_dynamics,flamingo,tng,uncertainty,2.0,12.478419,12.543357,0.064937,...,0.349487,2.351704,3.494272,5.258590,20,17.812506,16.634432,8.514285
6,flamingo_stellar_sum,flamingo,tng,nominal,0.5,12.475094,12.511238,0.036144,...,0.242911,2.458748,3.245178,4.301547,100000,92104.055144,92104.055144,92104.055144
7,flamingo_stellar_sum,flamingo,tng,nominal,1.0,12.475094,12.512420,0.037325,...,0.244672,2.456397,3.254016,4.314896,100000,92191.666142,92191.666142,92191.666142
8,flamingo_stellar_sum,flamingo,tng,nominal,2.0,12.475094,12.513697,0.038603,...,0.253547,2.431692,3.263601,4.359695,100000,92511.348891,92511.348891,92511.348891
9,flamingo_stellar_sum,flamingo,tng,uncertainty,0.5,12.478620,12.512369,0.033749,...,0.250273,2.451676,3.253635,4.362510,20,18.546317,16.818046,13.601082


### Box support and density families

The [11.5,14.5] box restricts sampling rather than setting the training mass frequency.
We widen, remove or tighten that support while keeping the learned conditional Gaussian
unchanged. Analytic checks are compared with the saved 500-draw results, including
finite Monte Carlo variability.

The MAF/NSF/MDN comparison covers earlier dynamics models and several seeds. It
does not establish that flexible density families are unnecessary for the stellar or
combined models. Those comparisons remain proposed follow-ups.


In [3]:
display(table('bounds/summary_across_models.csv'))
checks=table('bounds/validation_checks.csv')
assert checks.passed.all()
print('Saved support-sensitivity checks:',len(checks))
display(table('model_family/npe_model_family_predictive_metrics.csv'))
display(table('model_family/npe_model_family_feature_contrast.csv'))

Saved support-sensitivity checks: 262


,scenario,lower,upper,kind,lg_max_abs_median_change_dex,lg_max_abs_mass_change_percent,lg_max_abs_width68_change_percent,heldout_max_abs_bias_change_dex,heldout_max_abs_rmse_change_dex,heldout_max_abs_coverage68_change_percentage_points,heldout_max_individual_median_change_dex,max_training_rows_excluded,max_test_rows_excluded
0,original,11.50,14.50,baseline,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0,0
1,expand_lower,11.00,14.50,wider,6.117219e-06,1.408532e-03,2.343788e-03,0.001547,0.001496,0.733336,0.033895,0,0
2,expand_upper,11.50,15.00,wider,4.778400e-13,1.100267e-10,1.396105e-10,0.000035,0.000068,0.000000,0.016698,0,0
3,wide,11.00,15.00,wider,6.117219e-06,1.408532e-03,2.343788e-03,0.001524,0.001453,0.733336,0.033895,0,0
4,very_wide,10.00,16.00,wider,6.118304e-06,1.408782e-03,2.344179e-03,0.001603,0.001554,0.775375,0.036492,0,0
5,upper_14,11.50,14.00,modest_tightening,7.467431e-09,1.719440e-06,4.205282e-06,0.000528,0.000723,0.432900,0.153329,0,0
6,near_population,11.54,13.87,near_catalogue_range,4.865787e-06,1.120395e-03,1.962962e-03,0.000926,0.001379,0.865801,0.241851,0,0
7,lower_12,12.00,14.50,restrictive_stress,3.746949e-03,8.664995e-01,1.786331e+00,0.023390,0.015984,10.248027,0.254298,5091,985
8,upper_13,11.50,13.00,restrictive_stress,5.753482e-03,1.316052e+00,5.430638e+00,0.076119,0.062127,14.285714,1.022990,6921,1301
9,tight_12_13,12.00,13.00,restrictive_stress,3.247074e-03,7.448783e-01,5.430639e+00,0.075660,0.062060,15.151515,1.022990,12012,2286


,key,feature_set,model,seed,n_features,n_train,n_test,test_log_prob,...,rmse,bias,coverage68,coverage95,rank_cdf_distance,mean_width68,median_beta,elapsed_minutes
0,kinematics_only__maf__seed42,kinematics_only,maf,42,3,67713,11929,0.363287,...,0.185883,0.002948,0.707519,0.947858,0.030005,0.352572,0.635369,5.645307
1,kinematics_only__maf__seed43,kinematics_only,maf,43,3,67713,11929,0.362517,...,0.186404,0.010101,0.707771,0.947607,0.040287,0.354983,0.645914,9.978724
2,kinematics_only__maf__seed44,kinematics_only,maf,44,3,67713,11929,0.362366,...,0.186533,0.006538,0.708777,0.946517,0.031587,0.354749,0.637662,8.167927
3,kinematics_only__mdn__seed42,kinematics_only,mdn,42,3,67713,11929,0.405610,...,0.186221,0.000460,0.671054,0.938134,0.007582,0.335101,0.634938,3.195338
4,kinematics_only__mdn__seed43,kinematics_only,mdn,43,3,67713,11929,0.405492,...,0.186328,0.006846,0.681113,0.938302,0.020679,0.339381,0.625188,4.106366
5,kinematics_only__mdn__seed44,kinematics_only,mdn,44,3,67713,11929,0.405530,...,0.186228,0.006298,0.675329,0.939643,0.017682,0.336436,0.632797,3.513578
6,kinematics_only__nsf__seed42,kinematics_only,nsf,42,3,67713,11929,0.377415,...,0.186477,0.001570,0.670383,0.931511,0.017004,0.329246,0.623099,21.280528
7,kinematics_only__nsf__seed43,kinematics_only,nsf,43,3,67713,11929,0.376947,...,0.187038,-0.003365,0.666946,0.931595,0.010071,0.327165,0.638425,25.388724
8,kinematics_only__nsf__seed44,kinematics_only,nsf,44,3,67713,11929,0.377825,...,0.186461,0.000339,0.675497,0.934948,0.008073,0.333528,0.620586,18.501658
9,ratio_conditioned__maf__seed42,ratio_conditioned,maf,42,4,67713,11929,0.677575,...,0.131340,0.004396,0.670970,0.940314,0.028101,0.245852,0.701104,9.150303


,model,seed,scenario,conditioning,kinematics_only_median,ratio_conditioned_median,kinematics_only_width68,ratio_conditioned_width68,delta_median_ratio_minus_kinematics,width_ratio_conditioned_over_kinematics
0,maf,42,Salomon2021,marginalized,12.468636,12.368094,0.353183,0.318986,-0.100542,0.903173
1,maf,42,Salomon2021,point,12.476525,12.340600,0.344156,0.310773,-0.135925,0.903000
2,maf,42,vdM2012,point,12.454841,12.364549,0.277484,0.294304,-0.090292,1.060617
3,maf,43,Salomon2021,marginalized,12.478497,12.356649,0.363506,0.332166,-0.121848,0.913784
4,maf,43,Salomon2021,point,12.475972,12.317122,0.360402,0.332761,-0.158850,0.923304
5,maf,43,vdM2012,point,12.459743,12.357330,0.298179,0.305346,-0.102413,1.024035
6,maf,44,Salomon2021,marginalized,12.474070,12.358567,0.351797,0.324237,-0.115502,0.921659
7,maf,44,Salomon2021,point,12.478919,12.332998,0.351067,0.324341,-0.145921,0.923874
8,maf,44,vdM2012,point,12.447572,12.356585,0.278784,0.294365,-0.090987,1.055888
9,mdn,42,Salomon2021,marginalized,12.479534,12.355837,0.353093,0.337145,-0.123696,0.954833


Marginal support, empirical-prior shifts and conditional mapping differences are related
but distinct. Read these tests alongside common support and stellar normalization.
None selects a uniquely correct simulator.

The calculations are implemented in the [prior analysis](../research/supplements/prior/analyse_prior_shift.py),
[Local Group component reweighting](../research/supplements/prior/local_group_prior_swap.py),
[support sensitivity](../research/supplements/bounds/run_sensitivity.py), and
[density-family comparison](../research/notebooks/original/npe_model_family_comparison.ipynb).
The [reproduction notebook](92_reproduce_saved_analyses.ipynb) gives the input files
and order for rerunning the saved-draw analyses.


[Previous](10_common_support.ipynb) · [Next](12_local_group_and_interpretation.ipynb) · [Repository map](../README.md)
